In [13]:
import pandas as pd

In [36]:
order_payments = pd.read_csv("data/order_payments.csv")
products = pd.read_csv("data/products.csv")
order_items = pd.read_csv("data/order_items.csv")
customers = pd.read_csv("data/customers.csv")
orders= pd.read_csv("data/orders.csv")

In [38]:
# 1. TOTAL REVENUE
total_revenue = order_items["price"].sum()
print("Total Revenue:", total_revenue)

Total Revenue: 13591643.7


In [39]:
# 2. TOTAL ORDERS
total_orders = orders["order_id"].nunique()
print("Total Orders:", total_orders)

Total Orders: 99441


In [40]:
# 3. AVERAGE ORDER VALUE (AOV)
order_revenue = (
    order_items
    .groupby("order_id")["price"]
    .sum()
)
average_order_value = order_revenue.mean()
print("Average Order Value:", average_order_value)

Average Order Value: 137.75407637889444


In [ ]:
# 4. AVERAGE DELIVERY TIME
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)
orders["order_delivered_customer_date"] = pd.to_datetime(
    orders["order_delivered_customer_date"]
)
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days
average_delivery_time = orders["delivery_days"].mean()
print("Average Delivery Time:", average_delivery_time, "days")


Average Delivery Time: 12.093604229294082 days


In [43]:
# 5. CANCELLATION RATE
cancelled_orders = (
    orders["order_status"] == "canceled"
).sum()
cancellation_rate = (
    cancelled_orders / total_orders
) * 100
print("Cancellation Rate:", cancellation_rate, "%")


Cancellation Rate: 0.6285133898492574 %


In [44]:
# 6. REVENUE BY CATEGORY
sales_products = order_items.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)
revenue_by_category = (
    sales_products
    .groupby("product_category_name")["price"]
    .sum()
    .sort_values(ascending=False)
)
print("\nRevenue by Category:")
print(revenue_by_category.head(10))


Revenue by Category:
product_category_name
beleza_saude              1258681.34
relogios_presentes        1205005.68
cama_mesa_banho           1036988.68
esporte_lazer              988048.97
informatica_acessorios     911954.32
moveis_decoracao           729762.49
cool_stuff                 635290.85
utilidades_domesticas      632248.66
automotivo                 592720.11
ferramentas_jardim         485256.46
Name: price, dtype: float64


In [46]:
# 7. REVENUE BY STATE
orders_customers = orders.merge(
    customers[["customer_id", "customer_state"]],
    on="customer_id",
    how="left"
)
sales_state = order_items.merge(
    orders_customers[["order_id", "customer_state"]],
    on="order_id",
    how="left"
)
revenue_by_state = (
    sales_state
    .groupby("customer_state")["price"]
    .sum()
    .sort_values(ascending=False)
)
print("\nRevenue by State:")
print(revenue_by_state.head(10))


Revenue by State:
customer_state
SP    5202955.05
RJ    1824092.67
MG    1585308.03
RS     750304.02
PR     683083.76
SC     520553.34
BA     511349.99
DF     302603.94
GO     294591.95
ES     275037.31
Name: price, dtype: float64


In [49]:
# 8. MONTHLY REVENUE
orders["purchase_month"] = (
    orders["order_purchase_timestamp"]
    .dt.to_period("M")
)
sales_monthly = order_items.merge(
    orders[["order_id", "purchase_month"]],
    on="order_id",
    how="left"
)
monthly_revenue = (
    sales_monthly
    .groupby("purchase_month")["price"]
    .sum()
)
print("\nMonthly Revenue:")
print(monthly_revenue.head(10).sort_values(ascending=False))


Monthly Revenue:
purchase_month
2017-05    506071.14
2017-07    498031.48
2017-06    433038.60
2017-03    374344.30
2017-04    359927.23
2017-02    247303.02
2017-01    120312.87
2016-10     49507.66
2016-09       267.36
2016-12        10.90
Freq: M, Name: price, dtype: float64


In [ ]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Revenue",
        "Total Orders",
        "Average Order Value",
        "Average Delivery Time",
        "Cancellation Rate"
    ],
    "Value": [
        total_revenue,
        total_orders,
        average_order_value,
        average_delivery_time,
        cancellation_rate
    ]
})
print("\n================ KPI SUMMARY ================")
print(kpi_summary)


================ KPI SUMMARY ================
                     KPI         Value
0          Total Revenue  1.359164e+07
1           Total Orders  9.944100e+04
2    Average Order Value  1.377541e+02
3  Average Delivery Time  1.209360e+01
4      Cancellation Rate  6.285134e-01
